## 6개 모델 비교 예측
모든 실험(Exp1~6)의 모델을 로드하여 같은 입력에 대한 예측 결과 비교

In [ ]:
# Cell 0: 패키지 설치
!pip install sentence-transformers scikit-learn gensim torch -q

In [ ]:
# Cell 1: 패키지 임포트
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim.downloader as gensim_api
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Cell 2: MLP 클래스
class MLP(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, dropout_rate=0.0):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, output_size)
        self.activation = nn.GELU()
        self.output_act = nn.Softmax(dim=1)
        self.dropout = nn.Dropout(p=dropout_rate)
    
    def forward(self, x):
        x = self.dropout(self.activation(self.fc1(x)))
        x = self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
print('✅ MLP 클래스 준비')

In [ ]:
# Cell 3: 벡터화 도구 준비
# ⚠️ 학습 시 사용한 데이터로 fit 필요 (또는 저장된 vectorizer 로드)
from datasets import load_dataset

print('데이터 로드 중...')
data = load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
train_data = data['train'].filter(lambda r: all(r[f] not in [None,''] for f in ['text','label']))

# BoW
print('BoW 벡터화 준비...')
bow_vec = CountVectorizer()
bow_vec.fit(train_data['text'])

# TF-IDF
print('TF-IDF 벡터화 준비...')
tfidf_vec = TfidfVectorizer(max_features=30000)
tfidf_vec.fit(train_data['text'])

# GloVe
print('GloVe 로드 중...')
glove = gensim_api.load('glove-wiki-gigaword-100')
def to_glove(text):
    words = text.lower().split()
    vecs = [glove[w] for w in words if w in glove]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

# MiniLM
print('MiniLM 로드 중...')
minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# MPNet
print('MPNet 로드 중...')
mpnet = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

print('✅ 모든 벡터화 도구 준비 완료')

In [ ]:
# Cell 4: 6개 모델 설정
# ⚠️ Colab Files에 best_model_exp{1-6}.pt 업로드 필요

experiments = [
    {
        'name': 'Exp1 - BoW Baseline',
        'checkpoint': 'best_model_exp1.pt',
        'vectorizer': lambda text: bow_vec.transform([text]).toarray()[0],
        'input_size': len(bow_vec.vocabulary_),
        'hidden_size': 100,
        'dropout': 0.0
    },
    {
        'name': 'Exp2 - BoW + Sweep',
        'checkpoint': 'best_model_exp2.pt',
        'vectorizer': lambda text: bow_vec.transform([text]).toarray()[0],
        'input_size': len(bow_vec.vocabulary_),
        'hidden_size': 1000,
        'dropout': 0.2
    },
    {
        'name': 'Exp3 - TF-IDF + Sweep',
        'checkpoint': 'best_model_exp3.pt',
        'vectorizer': lambda text: tfidf_vec.transform([text]).toarray()[0],
        'input_size': tfidf_vec.max_features or len(tfidf_vec.vocabulary_),
        'hidden_size': 1000,
        'dropout': 0.2
    },
    {
        'name': 'Exp4 - GloVe + Sweep',
        'checkpoint': 'best_model_exp4.pt',
        'vectorizer': to_glove,
        'input_size': 100,
        'hidden_size': 1000,
        'dropout': 0.2
    },
    {
        'name': 'Exp5 - MiniLM + Sweep',
        'checkpoint': 'best_model_exp5.pt',
        'vectorizer': lambda text: minilm.encode([text], convert_to_numpy=True)[0],
        'input_size': 384,
        'hidden_size': 1000,
        'dropout': 0.2
    },
    {
        'name': 'Exp6 - MPNet + Sweep 🏆',
        'checkpoint': 'best_model_exp6.pt',
        'vectorizer': lambda text: mpnet.encode([text], convert_to_numpy=True)[0],
        'input_size': 768,
        'hidden_size': 1000,
        'dropout': 0.2
    }
]

print(f'✅ {len(experiments)}개 실험 설정 완료')

In [ ]:
# Cell 5: 모델 로드
models = []

for exp in experiments:
    try:
        # 모델 초기화
        model = MLP(exp['input_size'], exp['hidden_size'], 3, exp['dropout']).to(device)
        
        # 체크포인트 로드
        model.load_state_dict(torch.load(exp['checkpoint'], map_location=device))
        model.eval()
        
        models.append(model)
        print(f"✅ {exp['name']}: 로드 완료")
        
    except FileNotFoundError:
        models.append(None)
        print(f"❌ {exp['name']}: {exp['checkpoint']} 파일 없음")
    except Exception as e:
        models.append(None)
        print(f"❌ {exp['name']}: 오류 - {e}")

print(f'\n✅ 총 {sum(1 for m in models if m is not None)}/{len(models)}개 모델 로드 성공')

In [ ]:
# Cell 6: 전체 모델 비교 예측 함수
def compare_all_models(sentence):
    """같은 문장에 대해 6개 모델의 예측 결과 비교
    
    Args:
        sentence (str): 예측할 문장
    """
    print('\n' + '='*70)
    print(f'📝 Input: "{sentence}"')
    print('='*70)
    
    results = []
    
    for i, (exp, model) in enumerate(zip(experiments, models)):
        if model is None:
            results.append({
                'Experiment': exp['name'],
                'Prediction': 'N/A (model not loaded)',
                'Confidence': 0.0
            })
            continue
        
        try:
            # 벡터화
            vec = exp['vectorizer'](sentence)
            
            # Tensor 변환
            tensor = torch.FloatTensor(vec).unsqueeze(0).to(device)
            
            # 예측
            with torch.no_grad():
                output = model(tensor)
                probs = output[0].cpu().numpy()
                pred_class = int(np.argmax(probs))
                confidence = float(probs[pred_class])
            
            prediction = label_map[pred_class]
            
            results.append({
                'Experiment': exp['name'],
                'Prediction': prediction,
                'Confidence': f'{confidence*100:.1f}%'
            })
            
        except Exception as e:
            results.append({
                'Experiment': exp['name'],
                'Prediction': f'Error: {str(e)[:30]}',
                'Confidence': 'N/A'
            })
    
    # 결과 출력
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    print('='*70)
    
    # 예측 통계
    valid_preds = [r['Prediction'] for r in results if r['Prediction'] in label_map.values()]
    if valid_preds:
        from collections import Counter
        counts = Counter(valid_preds)
        majority = counts.most_common(1)[0]
        print(f'\n📊 예측 분포: {dict(counts)}')
        print(f'🏆 다수결: {majority[0]} ({majority[1]}/{len(valid_preds)} 모델)')

print('✅ 비교 함수 준비 완료')

In [ ]:
# Cell 7: 테스트 문장들
test_sentences = [
    "I love this item",
    "This is the worst product ever",
    "It was okay, nothing special",
    "Absolutely amazing experience!",
    "Terrible service, never coming back",
    "The quality is acceptable for the price"
]

for sentence in test_sentences:
    compare_all_models(sentence)

In [ ]:
# Cell 8: 사용자 입력 (Interactive)
print('💬 사용자 입력 모드 - 6개 모델 비교')
print('   종료: "quit" 입력\n')

while True:
    user_input = input('문장 입력: ').strip()
    
    if user_input.lower() == 'quit':
        print('종료합니다.')
        break
    
    if not user_input:
        print('⚠️ 문장을 입력해주세요.\n')
        continue
    
    compare_all_models(user_input)